# Customer Churn & Retention Intelligence — Exploratory Data Analysis

**Author:** Senior Data Scientist & ML Engineer  
**Dataset:** Bank Customer Churn (`Churn_Modelling.csv`)  
**Objective:** Perform exploratory data analysis to uncover customer churn drivers, evaluate data quality, assess class imbalance, and design feature engineering transformations.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.data.loader import load_raw_data
from src.data.validator import DataValidator
from src.features.engineering import engineer_features_df
from src.analytics.eda import generate_dataset_summary, calculate_segment_churn, calculate_financial_exposure

print('Environment and modules loaded successfully!')

## 1. Data Ingestion & Data Quality Audit
Load raw records and inspect missing values, duplicates, and column types.

In [2]:
df_raw = load_raw_data()
validator = DataValidator(df_raw)
quality_report = validator.run_full_validation()

print(f'Total records: {len(df_raw):,}')
print(f'Total features: {df_raw.shape[1]}')
print(f'Total missing values: {quality_report["summary"]["total_missing_values"]}')
print(f'Duplicate Customer IDs: {quality_report["summary"]["duplicate_customer_ids"]}')
df_raw.head()

## 2. Target Variable Analysis & Class Imbalance
Analyze the binary target `Exited` (1 = churned, 0 = retained).

In [3]:
summary = generate_dataset_summary(df_raw)
print(f'Retained Customers: {summary["retained_customers"]:,} ({summary["retention_rate"]*100:.2f}%)')
print(f'Churned Customers:  {summary["churned_customers"]:,} ({summary["churn_rate"]*100:.2f}%)')
print(f'Class Imbalance Ratio: ~{summary["retained_customers"]/summary["churned_customers"]:.2f}:1')

## 3. Segment Churn Drivers
Empirical churn rates across Geography, Gender, Products, and Activity.

In [4]:
print('--- Churn by Geography ---')
print(calculate_segment_churn(df_raw, 'Geography')[['Geography', 'TotalCustomers', 'ChurnedCustomers', 'ChurnRate']])

print('\n--- Churn by NumOfProducts ---')
print(calculate_segment_churn(df_raw, 'NumOfProducts')[['NumOfProducts', 'TotalCustomers', 'ChurnedCustomers', 'ChurnRate']])

print('\n--- Churn by Member Activity ---')
print(calculate_segment_churn(df_raw, 'IsActiveMember')[['IsActiveMember', 'TotalCustomers', 'ChurnedCustomers', 'ChurnRate']])

## 4. Financial Deposit Exposure
Differentiating customer deposits from revenue.

In [5]:
fin = calculate_financial_exposure(df_raw)
print(f'Total Deposits Held: ${fin["total_deposits"]:,.2f}')
print(f'Deposits of Churned Accounts: ${fin["churned_deposits"]:,.2f} ({fin["churned_deposits_share"]*100:.2f}%)')
print(f'Average Balance of Churned Customers: ${fin["avg_balance_churned"]:,.2f}')
print(f'Average Balance of Retained Customers: ${fin["avg_balance_retained"]:,.2f}')

## 5. Feature Engineering
Computing generational age cohorts, tenure buckets, and balance-to-salary ratios.

In [6]:
df_enriched = engineer_features_df(df_raw)
print('Engineered columns:', [c for c in df_enriched.columns if c not in df_raw.columns])
df_enriched[['CustomerId', 'AgeGroup', 'TenureGroup', 'BalanceSalaryRatio', 'CreditScoreAgeRatio', 'HasZeroBalance']].head()